# ラベル変換ガイド

`realign_external_labels` を使って、他のツールで作成したラベルファイルを
pyshiro モデルのアライメント基準に変換します。

## 同梱モデルのラベリング基準

同梱の `checkpoint/pyshiro-jp-v1.hsmm` は以下のラベリング基準で訓練されています。

### 母音の終了位置

母音の終了位置を、**波形が完全になくなる少し手前**に設定します。  
これにより、音響モデルがロングトーンのノート後半を無音として生成してしまう現象を防ぎます。

### 破裂音・破擦音の開始位置

波形の前に無音区間がある破裂音（k, t, p など）・破擦音（ch, ts）は、
**その無音区間（閉鎖区間）も含めて**ラベルを付けます。  
これにより、直前の母音が無音区間に食い込まないようにし、
ノートの後半が無声区間として生成される問題を抑制します。


## 仕組み

![concept](../doc/realign_concept.png)

**アンカー**（固定点）として信頼性が高い場所を外部ラベルから自動検出します：

- **長い母音**（デフォルト 0.5 秒以上）の中心時刻
- **長い pau**（デフォルト 0.5 秒以上）の両端から 0.25 秒の位置

隣接するアンカー間の区間（通常 30 秒以下）を HSMM で再アライメントします。

**`--fix_transitions`** で修正する音素境界タイプを絞れます。  
推奨設定 `vowel-consonant,silence-consonant,vowel-silence` を指定すると、
母音→子音・無音→子音・母音→無音の 3 種類の境界だけ修正し、
それ以外（子音→母音境界など）は元の外部ラベルを維持します。


## セットアップ

In [ ]:
import sys
from pathlib import Path

# pyshiro_pr のルートをパスに追加
ROOT = Path(".").resolve().parent
sys.path.insert(0, str(ROOT))

import pyshiro
from pyshiro.labels import read_lab, read_textgrid, write_lab_sec, write_textgrid
from pyshiro.align  import realign_external_labels, SILENCE_PHONES

print(f"pyshiro loaded from: {ROOT}")

## ファイルパスの設定

In [ ]:
WAV_PATH    = ROOT / "example/convert_labels/wav/01.wav"
LAB_PATH    = ROOT / "example/convert_labels/raw_lab/01.lab"
MODEL_PATH  = ROOT / "checkpoint/pyshiro-jp-v1.hsmm"
PHONEMAP    = ROOT / "checkpoint/pyshiro-jp-v1_phonemap.json"
OUT_DIR     = ROOT / "example/convert_labels/fixed_lab_recommended"
OUT_DIR.mkdir(exist_ok=True)

# ファイルの存在確認
for p in [WAV_PATH, LAB_PATH, MODEL_PATH, PHONEMAP]:
    print(f"{'OK' if p.exists() else 'MISSING'}: {p.name}")

## モデルとラベルの読み込み

In [ ]:
model    = pyshiro.load_hsmm(MODEL_PATH)
phonemap = pyshiro.load_phonemap(PHONEMAP)

# 外部ラベル読み込み（HTK 100ns / 秒単位を自動判定）
intervals = read_lab(LAB_PATH)   # [(start_sec, end_sec, phoneme), ...]

print(f"音素数: {len(intervals)}")
print(f"総時間: {intervals[-1][1]:.1f}s")
print("先頭 5 件:")
for s, e, ph in intervals[:5]:
    print(f"  {s:.4f}s - {e:.4f}s  [{ph}]  ({(e-s)*1000:.0f}ms)")

## 特徴量抽出

In [ ]:
streams = pyshiro.extract_mfcc_from_file(WAV_PATH)
T       = streams[0].shape[0]
print(f"T = {T} frames ({T * 0.005:.1f}s)")

## ラベル変換

### パターン A: デフォルト（全境界を修正）

In [ ]:
result_default = realign_external_labels(
    model, streams, intervals, phonemap,
    nodur_phonemes={"pau", "br", "cl"},
)
print(f"変換後音素数: {len(result_default)}")

### パターン B: 推奨設定

母音→子音・無音→子音・母音→無音の 3 種類の境界のみ修正します。

- **母音→子音**: 子音の閉鎖区間・無声区間を前方に延伸
- **無音→子音**: pau 後の子音開始境界を調整
- **母音→無音**: 母音の終了境界を調整（波形消滅点への追従）

子音→母音境界（= 母音の開始位置）は元のラベルを維持します。


In [ ]:
_ALL = frozenset(
    (a, b)
    for a in ("silence", "vowel", "consonant")
    for b in ("silence", "vowel", "consonant")
)
FIX = frozenset([
    ("vowel",   "consonant"),  # 母音→子音: 閉鎖区間を前方に延伸
    ("silence", "consonant"),  # 無音→子音: pau後の子音開始を調整
    ("vowel",   "silence"),    # 母音→無音: 母音終了境界を調整
])
IGNORE = _ALL - FIX  # 修正しない境界タイプ

result_recommended = realign_external_labels(
    model, streams, intervals, phonemap,
    nodur_phonemes={"pau", "br", "cl"},
    ignore_transitions=IGNORE,
)
print(f"変換後音素数: {len(result_recommended)}")


### 比較: 変換前後の確認

In [ ]:
# result はフレーム単位で返るので 0.005 をかけて秒に変換
res = result_recommended

# 元ラベルと並べて表示（最初の発話区間のみ）
VOWELS = set("aiueoAIUEON")

# 最初の長い pau の後から表示
start_idx = next(
    (i for i, (s, e, ph) in enumerate(intervals) if ph != "pau" and s > 1.0),
    0
)

print(f"{'音素':6s}  {'元ラベル':>24s}  {'変換後':>24s}  {'変化':>8s}")
print("-" * 70)
for i in range(start_idx, min(start_idx + 15, len(intervals))):
    s0, e0, ph = intervals[i]
    s1 = res[i][0] * 0.005
    e1 = res[i][1] * 0.005
    delta_s = (s1 - s0) * 1000
    mark = f"{delta_s:+.0f}ms" if abs(delta_s) > 1 else "-"
    print(f"{ph:6s}  {s0:.3f}-{e0:.3f}s ({(e0-s0)*1000:.0f}ms)  "
          f"{s1:.3f}-{e1:.3f}s ({(e1-s1)*1000:.0f}ms)  {mark}")

## 書き出し

In [ ]:
# 秒単位 lab として保存
write_lab_sec(result_recommended, OUT_DIR / "01_fixed.lab")
print("保存: example/convert_labels/fixed_lab_recommended/01_fixed.lab")

# Praat TextGrid として保存（手修正に便利）
write_textgrid(result_recommended, OUT_DIR / "01_fixed.TextGrid")
print("保存: example/convert_labels/fixed_lab_recommended/01_fixed.TextGrid")

## ラベルの外れ値チェック

変換したラベルの品質確認に、`workflow/05_check_consonant_outliers.py` を使います。
同一種類の子音のうち duration が外れ値のものを検出し、波形と音素ラベルを
1 枚の画像にまとめてプロットします。

**短すぎる子音はラベリングに失敗している可能性が高い**、という経験則に基づくツールです。
`--side short`（デフォルト）と `--side long` がありますが、実用上は短い側を見ることが
ほとんどです（長い側は閉鎖区間込みの正常な伸長が多く、失敗の信号になりにくい）。


In [ ]:
import subprocess
from IPython.display import Image, display

OUTLIER_PNG = ROOT / "example/convert_labels/outlier_plots/01_short_cons.png"
OUTLIER_PNG.parent.mkdir(parents=True, exist_ok=True)

# 短い側の外れ値を検出してプロット
subprocess.run([
    "python", str(ROOT / "workflow/05_check_consonant_outliers.py"),
    str(WAV_PATH),
    str(ROOT / "example/convert_labels/fixed_lab_recommended/01.lab"),
    "--side", "short",
    "--out", str(OUTLIER_PNG),
], check=True)

# 生成された図を表示
display(Image(filename=str(OUTLIER_PNG)))

赤くハイライトされた子音が「短い側の外れ値」です。  
比率（`Nx median`）が大きいものほど、その音素の中央値に対して極端に短く、
ラベリング失敗の疑いが強くなります。波形を見て、実際に子音が短いのか、
境界がずれているのかを確認します。

主なオプション:

| オプション | デフォルト | 説明 |
|---|---|---|
| `--side` | `short` | 検出する側（`short` / `long`） |
| `--floor_ms` | `40` | [short] この長さ（ms）未満は無条件でフラグ |
| `--max_dur_ms` | `50` | [short] この長さ未満かつ統計的外れ値を検出 |
| `--iqr_k` | `1.5` | 通常子音の IQR 倍率 |
| `--plosive_iqr_k` | `3.0` | 破裂音・破擦音の IQR 倍率（閉鎖込みで二峰性のため大きめ） |


## コマンドライン版

上記と同等の処理をコマンドラインで実行できます。

```bash
python workflow/04_convert_labels.py \\
    example/convert_labels/wav/01.wav \\
    example/convert_labels/raw_lab/01.lab \\
    --model    checkpoint/pyshiro-jp-v1.hsmm \\
    --phonemap checkpoint/pyshiro-jp-v1_phonemap.json \\
    --out      example/convert_labels/fixed_lab_recommended/01.lab \\
    --format   lab_sec \\
    --fix_transitions vowel-consonant,silence-consonant,vowel-silence
```

### オプション早見表

| オプション | デフォルト | 説明 |
|---|---|---|
| `--format` | `lab` | `lab` / `lab_sec` / `textgrid` / `audacity` |
| `--fix_transitions` | （全修正） | 修正する境界タイプ（推奨: `vowel-consonant,silence-consonant,vowel-silence`） |
| `--anchor_vowel_min` | `0.5` | アンカーにする母音の最小長（秒） |
| `--anchor_pau_min` | `0.5` | アンカーにする pau の最小長（秒） |
| `--max_seg_sec` | `30.0` | アンカー間の最大処理幅（秒） |
| `--phoneme_map` | — | 音素名変換 JSON（例: `{"q": "cl", "sil": "pau"}`） |
| `--unknown_phoneme_map` | — | 未知音素の代替（例: `vy=v,fy=f`）。出力は元の音素名 |
| `--silence_phones` | `pau,sil,cl,br` | 無音クラスの音素 |
| `--no_refine` | — | pau/子音境界のエネルギーオンセット補正を無効化 |


## 音素クラスと境界タイプ

`--fix_transitions` で指定する `FROM-TO` の各クラスの内訳：

| クラス | 音素 |
|---|---|
| `silence` | `pau`, `sil`, `cl`, `br`（`--silence_phones` で変更可） |
| `vowel` | `a`, `i`, `u`, `e`, `o`, `N`, `A`, `I`, `U`, `E`, `O` |
| `consonant` | 上記以外の全音素 |

全 9 種の境界タイプ：

```
silence-silence    silence-vowel    silence-consonant
vowel-silence      vowel-vowel      vowel-consonant
consonant-silence  consonant-vowel  consonant-consonant
```

未知音素（phonemap に未登録）はデフォルトでアンカー扱い（境界固定）になります。
`--unknown_phoneme_map vy=v,fy=f` のように代替音素を指定すると、アライメント時のみ代替し、出力ラベルは元の音素名を維持します。